# The Ceiling Is in the Signature

### Copy is not forbidden. A conjunction of four requirements on Copy is.

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## The criterion was too loose, and the fix makes the real question computable

The last notebook proposed: *a requirement is a price if a resource appears in its statement, a ceiling if
none does.* That is a useful way to **find** candidates and a bad way to **decide** them, for exactly the
reason the programme already demonstrated on itself. The bound $p\ge3$ on dynamical order contained no
$\tau$ — and dissolved the moment delay was admitted. Absence from one statement of a law is absence from one
*class*, not from physics.

The durable version:

$$\boxed{\;\text{Price: enlarging the resource space keeps moving the bound.}\;}$$
$$\boxed{\;\text{Floor: resources improve the bound, but a class-invariant minimum remains.}\;}$$
$$\boxed{\;\text{Ceiling: enlarging the resource space leaves the admissibility boundary where it was.}\;}$$

Which turns the Copy question from a claim into an experiment. Last time I spent one resource — ancilla
dimension — and the fidelity saturated at $5/6$. One resource is not the resource space. So this notebook
spends the rest of them.

The specification tested last time was not "Copy." It was

```text
Copy(exact=True, deterministic=True, universal=True, symmetric=True, inputs=1)
```

Five flags, and the $5/6$ result holds them all at once. **Each one is separately relaxable, and each
relaxation is a resource extension the criterion says to try.** If the ceiling is a property of the verb, none
of them should help. If it is a property of the specification, each should have its own published rate.

---

## Step 1 — Library


In [1]:
import numpy as np
from scipy.optimize import minimize, brentq

# The six octahedron states are an exact 2-design for a qubit, so averaging fidelity
# over them equals the Haar average exactly.
SIX = [np.array([1, 0]), np.array([0, 1]),
       np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2),
       np.array([1, 1j]) / np.sqrt(2), np.array([1, -1j]) / np.sqrt(2)]

def U_of(p, d):
    """A unitary from a real parameter vector, via a Hermitian generator."""
    n = d * d
    Hm = p[:n].reshape(d, d) + 1j * p[n:2*n].reshape(d, d)
    Hm = Hm + Hm.conj().T
    ev, V = np.linalg.eigh(Hm)
    return V @ np.diag(np.exp(1j * ev)) @ V.conj().T

def both_fidelities(U, da):
    """Fidelity of EACH output marginal against the input, Haar-averaged.
       Scoring only one output measures preservation, not copying: the identity
       channel would score 1. (Correction carried over from the last notebook.)"""
    F1 = F2 = 0.0
    for psi in SIX:
        v = np.kron(np.kron(psi, np.array([1, 0])), np.eye(da)[0])
        T = (U @ v).reshape(2, 2, da)
        r1 = np.einsum('ijk,ljk->il', T, T.conj())      # trace out clone 2 and ancilla
        r2 = np.einsum('jik,jlk->il', T, T.conj())      # trace out clone 1 and ancilla
        F1 += np.real(psi.conj() @ r1 @ psi)
        F2 += np.real(psi.conj() @ r2 @ psi)
    return F1 / len(SIX), F2 / len(SIX)

def best_over_unitaries(objective, d, da, restarts=5, seed=0, maxiter=800):
    """Maximize an objective of (F1, F2) over unitaries on 2 x 2 x da."""
    rng = np.random.default_rng(seed); best = None
    for _ in range(restarts):
        p0 = rng.normal(0, 0.6, 2 * d * d)
        r = minimize(lambda p: -objective(*both_fidelities(U_of(p, d), da)),
                     p0, method='L-BFGS-B', options={'maxiter': maxiter, 'ftol': 1e-14})
        f = both_fidelities(U_of(r.x, d), da)
        if best is None or objective(*f) > objective(*best):
            best = f
    return best


# ===========================================================================
# Duan-Guo: exact PROBABILISTIC cloning of a known state set is possible with
# success probabilities p_i iff   X1 - sqrt(Gam) X2 sqrt(Gam) >= 0
#   X1_ij = <psi_i|psi_j>,   X2_ij = <psi_i|psi_j>^2,   Gam = diag(p_i)
# For two states with overlap x and equal p this is a 2x2 positivity condition.
# ===========================================================================
def duan_guo_min_eig(x, p):
    X1 = np.array([[1.0, x], [x, 1.0]])
    X2 = np.array([[1.0, x * x], [x * x, 1.0]])
    return np.linalg.eigvalsh(X1 - p * X2).min()

def duan_guo_boundary(x):
    """Largest feasible p, located numerically -- not told the answer."""
    f = lambda p: duan_guo_min_eig(x, p)
    return brentq(f, 1e-12, 1.0) if f(1.0) < 0 else 1.0


# ===========================================================================
# Optimal state estimation from N copies. The covariant POVM
#   E(phi) = (N+1) |phi^N><phi^N| dphi
# is valid on the symmetric subspace, and gives
#   F = (N+1) * Integral |<phi|psi_0>|^(2N+2) dphi
# ===========================================================================
def estimation_fidelity_mc(N, M=400000, seed=0):
    rng = np.random.default_rng(seed)
    v = rng.normal(size=(M, 4)); v /= np.linalg.norm(v, axis=1, keepdims=True)
    phi = v[:, :2] + 1j * v[:, 2:]; phi /= np.linalg.norm(phi, axis=1, keepdims=True)
    ov = np.abs(phi[:, 0]) ** 2                         # |<phi|0>|^2, Haar distributed
    return (N + 1) * np.mean(ov ** (N + 1))

print("library ready")


library ready


---
## Step 2 — Relax universality

The weakest flag first. If the state set is known and orthogonal, CNOT is an exact, deterministic cloner. No
approximation, no failure probability, nothing spent.


In [2]:
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], float)
print("Exact deterministic cloning of a known orthogonal pair:")
for lbl, psi in [("|0>", np.array([1.,0])), ("|1>", np.array([0,1.]))]:
    err = np.linalg.norm(CNOT @ np.kron(psi, np.array([1.,0])) - np.kron(psi, psi))
    print(f"   {lbl:>4}:  || U|psi,0> - |psi,psi> ||  =  {err:.2e}")
sup = np.array([1.,1]) / np.sqrt(2)
err = np.linalg.norm(CNOT @ np.kron(sup, np.array([1.,0])) - np.kron(sup, sup))
print(f"   but on (|0>+|1>)/sqrt2:                   =  {err:.4f}")
print("\n   Free for a known orthogonal set. Fails the instant the set is enlarged.")
print("   Universality is a conjunct, and its price is zero when you drop it.")


Exact deterministic cloning of a known orthogonal pair:
    |0>:  || U|psi,0> - |psi,psi> ||  =  0.00e+00
    |1>:  || U|psi,0> - |psi,psi> ||  =  0.00e+00
   but on (|0>+|1>)/sqrt2:                   =  0.7654

   Free for a known orthogonal set. Fails the instant the set is enlarged.
   Universality is a conjunct, and its price is zero when you drop it.


---
## Step 3 — Relax determinism

Allow the copier to fail and announce the failure. Duan–Guo gives the feasibility condition for *exact*
probabilistic cloning of a known set, and for two states at overlap $x$ with equal success probability it is a
$2\times2$ positivity condition. Solving it by hand gives $p\le 1/(1+x)$ — so locate the boundary numerically
without telling the code the answer, and see where it lands.


In [3]:
print("Exact cloning, allowed to fail. Boundary located by root-finding on the")
print("Duan-Guo positivity condition -- the closed form is only used to check.\n")
print(f"{'overlap x':>10} | {'min eig at 0.98p':>17} {'at 1.02p':>11} | {'boundary found':>15} {'1/(1+x)':>10}")
print("-" * 74)
ok = tot = 0
for x in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    ps = 1 / (1 + x)
    lo = duan_guo_min_eig(x, 0.98 * ps)
    hi = duan_guo_min_eig(x, min(1.02 * ps, 1.0))
    b = duan_guo_boundary(x)
    good = abs(b - ps) < 1e-9; ok += good; tot += 1
    print(f"{x:>10} | {lo:>17.3e} {hi:>11.3e} | {b:>15.9f} {ps:>10.6f}  {'ok' if good else 'MISS'}")
print(f"\n   {ok} of {tot} boundaries land on 1/(1+x).")
print("   x = 0    -> p = 1:   orthogonal states clone exactly and deterministically.")
print("   x -> 1   -> p -> 1/2.")
print("\n   Exactness is purchasable. The currency is success probability, and the rate is 1/(1+x).")


Exact cloning, allowed to fail. Boundary located by root-finding on the
Duan-Guo positivity condition -- the closed form is only used to check.

 overlap x |  min eig at 0.98p    at 1.02p |  boundary found    1/(1+x)
--------------------------------------------------------------------------
       0.0 |         2.000e-02   0.000e+00 |     1.000000000   1.000000  ok
       0.1 |         1.800e-02  -1.800e-02 |     0.909090909   0.909091  ok
       0.3 |         1.400e-02  -1.400e-02 |     0.769230769   0.769231  ok
       0.5 |         1.000e-02  -1.000e-02 |     0.666666667   0.666667  ok
       0.7 |         6.000e-03  -6.000e-03 |     0.588235294   0.588235  ok
       0.9 |         2.000e-03  -2.000e-03 |     0.526315789   0.526316  ok
      0.99 |         2.000e-04  -2.000e-04 |     0.502512563   0.502513  ok

   7 of 7 boundaries land on 1/(1+x).
   x = 0    -> p = 1:   orthogonal states clone exactly and deterministically.
   x -> 1   -> p -> 1/2.

   Exactness is purchasable. The

Note what this does to the exact-cloning argument. The impossibility proof runs
$x=\langle\psi|\phi\rangle=x^2$, with no resource in it — and that is still true. But the proof assumed a
*deterministic* unitary on the whole space. Admit a flag qubit and postselection and the obstruction becomes a
*rate*: $1/(1+x)$, continuous in the overlap, reaching 1 exactly where the states go orthogonal.

The resource was not absent from physics. It was absent from the class the statement quantified over.

---
## Step 4 — Relax symmetry

Now the flag that looks least like a resource. Require the two outputs to be equally good, and the optimum is
$5/6$ each. Drop that and maximize a weighted sum instead, tracing the frontier.


In [4]:
print("Frontier of the asymmetric cloner (ancilla dim 2).")
print("Maximizing w*F1 + (1-w)*F2 over unitaries.\n")
print(f"{'weight w':>9} {'F1':>11} {'F2':>11} {'mean':>11}")
print("-" * 46)
for w in [0.5, 0.7, 0.85, 0.95, 1.0]:
    f1, f2 = best_over_unitaries(lambda a, b: w * a + (1 - w) * b, d=8, da=2, seed=3)
    print(f"{w:>9} {f1:>11.6f} {f2:>11.6f} {(f1+f2)/2:>11.6f}")
print(f"\n   Symmetric point: F1 = F2 = {5/6:.6f} = 5/6, as before.")
print("   At w = 1: F1 -> 1.000000 and F2 -> 0.500000, which is the no-information value.")
print("   That limit is just 'keep the original and hand over noise' -- a perfect clone,")
print("   bought by giving up the other one entirely. Symmetry is a conjunct.")


Frontier of the asymmetric cloner (ancilla dim 2).
Maximizing w*F1 + (1-w)*F2 over unitaries.

 weight w          F1          F2        mean
----------------------------------------------
      0.5    0.833332    0.833334    0.833333
      0.7    0.968063    0.639270    0.803667
     0.85    0.995412    0.550027    0.772719
     0.95    0.999635    0.513680    0.756658
      1.0    1.000000    0.499999    0.749999

   Symmetric point: F1 = F2 = 0.833333 = 5/6, as before.
   At w = 1: F1 -> 1.000000 and F2 -> 0.500000, which is the no-information value.
   That limit is just 'keep the original and hand over noise' -- a perfect clone,
   bought by giving up the other one entirely. Symmetry is a conjunct.


---
## Step 5 — Relax the input count

Give the copier $N$ copies of the unknown state instead of one. The optimal covariant estimation strategy
(the $M\to\infty$ limit of cloning) has fidelity $(N+1)/(N+2)$, which I can integrate directly over the Bloch
sphere rather than assert.


In [5]:
print("Optimal estimation fidelity from N copies, by Monte Carlo over Haar qubits:\n")
print(f"{'N':>5} {'Monte Carlo':>14} {'(N+1)/(N+2)':>14} {'difference':>12}")
print("-" * 50)
for N in [1, 2, 3, 5, 10, 50]:
    mc = estimation_fidelity_mc(N); ex = (N + 1) / (N + 2)
    print(f"{N:>5} {mc:>14.6f} {ex:>14.6f} {abs(mc-ex):>12.2e}")
print("\n   F -> 1 as N grows, with no floor. Input count is an ordinary price.")
print("   (At N=50 the Monte Carlo drifts: |<phi|psi>|^102 concentrates near 1 and the")
print("    uniform sampler undersamples that peak. Sampling error, not a failure of the law --")
print("    the same 'audit the reader' pattern as the earlier notebooks.)")


Optimal estimation fidelity from N copies, by Monte Carlo over Haar qubits:

    N    Monte Carlo    (N+1)/(N+2)   difference
--------------------------------------------------
    1       0.666572       0.666667     9.45e-05
    2       0.749949       0.750000     5.08e-05
    3       0.799986       0.800000     1.38e-05
    5       0.857188       0.857143     4.49e-05
   10       0.916896       0.916667     2.30e-04
   50       0.985819       0.980769     5.05e-03

   F -> 1 as N grows, with no floor. Input count is an ordinary price.
   (At N=50 the Monte Carlo drifts: |<phi|psi>|^102 concentrates near 1 and the
    uniform sampler undersamples that peak. Sampling error, not a failure of the law --
    the same 'audit the reader' pattern as the earlier notebooks.)


---
## Step 6 — What does *not* move

Four flags, four rates. Now hold all of them and spend the one resource that is not a conjunct: room to work
in. This is the test from the last notebook, repeated because it is the control.


In [6]:
print("Exact = False, deterministic, universal, symmetric, one input.")
print("Spend ancilla dimension -- the only resource left that is not one of the flags.\n")
print(f"{'ancilla dim':>12} {'total dim':>10} {'params':>8} {'best symmetric fidelity':>25} {'5/6':>10}")
print("-" * 70)
for da in [1, 2, 4]:
    d = 4 * da
    f1, f2 = best_over_unitaries(lambda a, b: 0.5 * (a + b), d=d, da=da, seed=0, restarts=5)
    print(f"{da:>12} {d:>10} {2*d*d:>8} {0.5*(f1+f2):>25.8f} {5/6:>10.6f}")
print("""
   Dimension 1 -> 2 buys the last of the approximation: 0.819 -> 0.8333.
   Dimension 2 -> 4 buys nothing. The bound is 5/6 and it stays there.
""")


Exact = False, deterministic, universal, symmetric, one input.
Spend ancilla dimension -- the only resource left that is not one of the flags.

 ancilla dim  total dim   params   best symmetric fidelity        5/6
----------------------------------------------------------------------
           1          4       32                0.81903559   0.833333
           2          8      128                0.83333333   0.833333
           4         16      512                0.83333332   0.833333

   Dimension 1 -> 2 buys the last of the approximation: 0.819 -> 0.8333.
   Dimension 2 -> 4 buys nothing. The bound is 5/6 and it stays there.



---
## Step 7 — The result: ceilings live at conjunctions

Every flag has a rate. The conjunction has a bound.

| Specification | Best achievable | Rate of the relaxation |
|---|---|---|
| all five flags held | $F=5/6$ | immovable by ancilla dimension |
| drop **universal** | exact, deterministic | free — CNOT |
| drop **deterministic** | exact | $p=1/(1+x)$ |
| drop **symmetric** | $F_1\to1$ | $F_2\to1/2$, the no-information value |
| drop **inputs $=1$** | $F=(N+1)/(N+2)\to1$ | one more copy at a time |

So the ceiling is not a property of the verb Copy. **It is a property of the signature.** `Copy()` is not
forbidden; `Copy(exact, deterministic, universal, symmetric, inputs=1)` is, and every argument you unset opens a
lawful realization with a published price.

That is a considerably better answer than "Copy is categorically excluded," and it is the one the programme's
own language predicted. This thread began with `Sun()` versus `Sun(a,b,c,d)` — the claim that learning exposes
arguments that were already callable. The Copy result is that claim running in reverse: **an impossibility is
also located in a signature**, and finding which argument carries it is the same work as finding which argument
carries a capability.

$$\boxed{\;\text{A ceiling is a conjunction. Name the conjuncts and each one has a rate.}\;}$$

The honest standing of the invariant: $x=x^2$ survives every extension tested here, because each extension
changed the specification rather than the algebra. What no test in this notebook can establish is that *no*
admissible extension touches it. The criterion says ceiling; a criterion is not a proof; and the programme has
already watched one apparently absolute gate dissolve. So the claim is: **invariant under every resource
extension tried, and the tried set now includes all four conjuncts plus working room.**

---
## Step 8 — The full taxonomy, five notebooks in

| Requirement | Category | The number |
|---|---|---|
| Latch onset | price | $\prod(G_i/\gamma_i)=1$ |
| Clock onset, cascade | price | $\sec(\pi/N)$ |
| Clock onset, delay | price | $\tau$ at $\arccos(1/G)/\sqrt{G^2-1}$ per stage |
| Clock onset, resonance | price | $2/Q$, and $\to0$ |
| Clock onset, diffusion | **floor** | $e^{\pi\sqrt{1+\mathrm{Da}}}\ \ge e^\pi$ within parabolic transport |
| Dynamical order $\ge3$ | price, class-relative | dissolved by delay |
| Hold, conserved mass | price | $M_c$ |
| Noise as a clock | price that loses | $\mathrm{CV}\sim\Omega^{-1/2}$ |
| Copy, four flags held | **ceiling at a conjunction** | $5/6$ |
| Copy, any one flag dropped | price | free, $1/(1+x)$, $F_2\to1/2$, $(N+1)/(N+2)$ |

Three categories, and the two non-price entries have opposite characters. **Diffusion's floor is inside a
class** — leave parabolic transport for the telegrapher family and the floor slides continuously to 1.
**Copy's ceiling is inside a specification** — leave any one conjunct and it becomes a rate. In both cases the
thing that looked absolute was absolute *relative to something*, and naming that something is the whole content.

---
## Step 9 — Corrections accepted

**The criterion, restated.** "No resource in the statement" is a discovery heuristic. The criterion is
invariance under resource extension, and this notebook is what applying it looks like: enumerate the
specification's conjuncts, relax each, measure the rate.

**"No larger class to escape into"** was too strong, and I withdraw it. The correct statement is that the
obstruction is invariant under the extensions tried. Which extensions count as spending a resource, and which
count as changing the transformation class, is the open question — and it is the sharper one, because
polymorphism is precisely the discovery that a new interface exposes a new lawful method.

**Method rule, now with four instances.** The programme's errors have all been caught the same way: mis-indexed
steady states, a biased sampler, a principal-branch phase, and a one-sided fidelity that made the identity a
perfect cloner. None were found by reading code. All were found by a number that could not be right.

$$\boxed{\;\text{When output is too good, too absolute, or violates a known limit: audit the reader before revising the world.}\;}$$

And its companion, the run-length rule: near any bifurcation the decay rate vanishes by construction, so
integration time must scale with $1/\lvert\mathrm{Re}\,\lambda_{\max}\rvert$ or a slow transient will pass for a
cycle.

---
## Step 10 — Still open

- **Which extensions are resources and which are class changes?** This is now the central question rather
  than a caveat. Adding delay felt like a resource and behaved like a class change. Postselection is the same
  ambiguity in the Copy case: it dissolved exactness, but it also changed what counts as an operation.
- **Erase.** The criterion predicts Landauer's $kT\ln2$ is a price because $T$ is in it, and it is — the bound
  goes to zero with temperature. That is a check the criterion passed unprompted, and the interesting version
  is whether the $T\to0$ limit hides a floor the way diffusion does.
- **Global structure**, still unaddressed across six notebooks. Every threshold is local, and the coexisting
  latch-and-cycle from the second notebook is still the standing counterexample to treating local conditions as
  a full account.
